In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress
from statsmodels.stats.stattools import durbin_watson

In [ ]:
# ==============================================================================
# 1. PARÁMETROS Y CONSTANTES
# ==============================================================================
L = 0.257         # Longitud de la barra (m)
x_sensor = np.array([0.05, 0.10, 0.15])  # Posición de los sensores (m)
sigma_T = 0.1     # precisión del sensor (°C)
T_A = 15       # temperatura ambiente (°C)

# Propiedades teóricas del hierro
k_m = 80.0        # W / m K (conductividad térmica de la barra)
rho_m = 7870.0    # kg / m^3 (densidad)
Cp_m = 460.0      # J / kg K (capacidad calorífica a presión constante)
alfa_teorico = k_m / (rho_m * Cp_m)

# Pérdidas laterales teóricas
r_int, r_ext = 0.00582,0.0125 #(radio interno y externo de la barra)
k_A = 0.04 #(conductividad térmica del aislante)
h_conv, h_rad = 8.49, 1.35 #(coeficientes de transferencia de calor por convección y radiación)
A_m = np.pi * r_int**2 #(área de sección transversal)

R_cond = np.log(r_ext / r_int) / (2 * np.pi * k_A) #(resistencia térmica de conducción)
R_paralelo = 1.0 / ((h_conv + h_rad) * 2 * np.pi * r_ext) #(resistencia térmica de convección y radiación)
R_total_estrella = R_cond + R_paralelo
beta_teorico = 1.0 / (rho_m * Cp_m * A_m * R_total_estrella)

k1 = np.pi / (2 * L)
k2 = 3 * np.pi / (2 * L)

# ==============================================================================
# 2. CARGA DE DATOS
# ==============================================================================
df = pd.read_csv('Libro1.csv', comment='#', header=None, names=['tiempo', 'posicion', 'temperatura'])
df['tiempo'] = pd.to_numeric(df['tiempo'], errors='coerce')
df['posicion'] = pd.to_numeric(df['posicion'], errors='coerce')
df['temperatura'] = pd.to_numeric(df['temperatura'], errors='coerce')
df = df.dropna()
# Pivoteo de la tabla(reordena la tabla de tal manera que cada fila sea de tiempo y cada columna la posición)
df_pivot = df.pivot(index='tiempo', columns='posicion', values='temperatura').reset_index()

t_c = df_pivot['tiempo'].values.astype(float)

# Matriz u(x_i, t_c) = T_medida - T_A
# Selección de columnas correspondientes a 0.05, 0.10 y 0.15 m
U_medido = df_pivot[[0.05, 0.10, 0.15]].values - T_A

# ==============================================================================
# 3. MÍNIMOS CUADRADOS PARA D_1(t_c) Y D_2(t_c)
# ==============================================================================
M = np.array([
    [np.sin(k1 * x_sensor[0]), np.sin(k2 * x_sensor[0])],
    [np.sin(k1 * x_sensor[1]), np.sin(k2 * x_sensor[1])],
    [np.sin(k1 * x_sensor[2]), np.sin(k2 * x_sensor[2])]
])

M_pseudo_inv = np.linalg.pinv(M)

#Producto M_pseudo_inv * U(t_c) 
D_matrix = np.zeros((len(t_c), 2))
for i in range(len(t_c)):
    D_matrix[i, :] = M_pseudo_inv @ U_medido[i, :]

D1, D2 = D_matrix[:, 0], D_matrix[:, 1]

# ==============================================================================
# 4. CRITERIOS Y SELECCIÓN DE VENTANA
# ==============================================================================

#Exclusión de regiones en donde las amplitudes caen por debajo de 5 veces la precisión del sensor. 
piso_ruido = 5 * sigma_T
ruido_1 = np.abs(D1) > piso_ruido
ruido_2 = np.abs(D2) > piso_ruido

#Se busca en cada intervalo de tiempo, las regresiones que maximizen el coeficiente de determinación.
def encontrar_ventana_optima(t, D, r):
    t_val = t[r]
    D_val = D[r]
    ln_D = np.log(np.abs(D_val))
    
    mejor_r2 = -1
    idx_inicio_opt = 0
    idx_fin_opt = len(t_val)
    
    min_puntos = 10
    for i in range(0, len(t_val) - min_puntos, 2):
        for j in range(i + min_puntos, len(t_val), 2):
            res = linregress(t_val[i:j], ln_D[i:j])
            if res.rvalue**2 > mejor_r2:
                mejor_r2 = res.rvalue**2
                idx_inicio_opt, idx_fin_opt = i, j
                
    return t_val[idx_inicio_opt:idx_fin_opt], ln_D[idx_inicio_opt:idx_fin_opt]

t_v1, ln_D1_v = encontrar_ventana_optima(t_c, D1, ruido_1)
t_v2, ln_D2_v = encontrar_ventana_optima(t_c, D2, ruido_2)

# ==============================================================================
# 5. REGRESIONES, PRUEBA DURBIN-WATSON(1,5 < Dw <  2,5) Y ERRORES
# ==============================================================================
res_1 = linregress(t_v1, ln_D1_v)
gamma_1 = -res_1.slope
sigma_gamma1 = res_1.stderr
residuos_1 = ln_D1_v - (res_1.intercept + res_1.slope * t_v1)
dw_1 = durbin_watson(residuos_1)

res_2 = linregress(t_v2, ln_D2_v)
gamma_2 = -res_2.slope
sigma_gamma2 = res_2.stderr
residuos_2 = ln_D2_v - (res_2.intercept + res_2.slope * t_v2)
dw_2 = durbin_watson(residuos_2)

delta_k = k2**2 - k1**2
alfa_exp = (gamma_2 - gamma_1) / delta_k
beta_exp = gamma_1 - alfa_exp * (k1**2)

# Propagación de Errores
sigma_alfa = np.sqrt(sigma_gamma1**2 + sigma_gamma2**2) / delta_k
term1 = (1.0 - (k1**2 / delta_k)) * sigma_gamma1
term2 = (k1**2 / delta_k) * sigma_gamma2
sigma_beta = np.sqrt(term1**2 + term2**2)

# ==============================================================================
# 6.RESULTADOS
# ==============================================================================
print(f"Temperatura Ambiente T_A: {T_A} °C")
print(f"Pendiente Modo 1 (gamma_1): {gamma_1:.6e} ± {sigma_gamma1:.6e} s^-1 (R^2 = {res_1.rvalue**2:.4f})")
print(f"Estadístico Durbin-Watson (Modo 1): {dw_1:.3f}")
print(f"Pendiente Modo 2 (gamma_2): {gamma_2:.6e} ± {sigma_gamma2:.6e} s^-1 (R^2 = {res_2.rvalue**2:.4f})")
print(f"Estadístico Durbin-Watson (Modo 2): {dw_2:.3f}")
print("----------------------------------------------------------------------")
print(f"Difusividad térmica (Alpha Exp):  {alfa_exp:.4e} ± {sigma_alfa:.4e} m^2/s")
print(f"Difusividad térmica (Alpha Teor): {alfa_teorico:.4e} m^2/s")
print("----------------------------------------------------------------------")
print(f"Coef. Pérdida Lateral (Beta Exp):  {beta_exp:.4e} ± {sigma_beta:.4e} s^-1")
print(f"Coef. Pérdida Lateral (Beta Teor): {beta_teorico:.4e} s^-1")
print("======================================================================\n")

# Gráficos
fig, axs = plt.subplots(1, 3, figsize=(14, 5))

axs[0].plot(t_v1, ln_D1_v, 'bo', markersize=3, label='ln|D1|')
axs[0].plot(t_v1, res_1.intercept + res_1.slope * t_v1, 'r-', label=f'gamma_1={gamma_1:.2e}')
axs[0].plot(t_v2, ln_D2_v, 'go', markersize=3, label='ln|D2|')
axs[0].plot(t_v2, res_2.intercept + res_2.slope * t_v2, 'm-', label=f'gamma_2={gamma_2:.2e}')
axs[0].set_xlabel('Tiempo $t_c$ (s)')
axs[0].set_ylabel('$\ln|D_n(t_c)|$')
axs[0].set_title('Decaimiento Modal')
axs[0].grid(True)
axs[0].legend()

axs[1].scatter(t_v1, residuos_1, color='purple', s=10)
axs[1].axhline(0, color='black', linestyle='--')
axs[1].set_xlabel('Tiempo $t_c$ (s)')
axs[1].set_ylabel('Residuos')
axs[1].set_title(f'Residuos Modo 1 (DW: {dw_1:.2f})')
axs[1].grid(True)


axs[2].scatter(t_v2, residuos_2, color='purple', s=10)
axs[2].axhline(0, color='black', linestyle='--')
axs[2].set_xlabel('Tiempo $t_c$ (s)')
axs[2].set_ylabel('Residuos')
axs[2].set_title(f'Residuos Modo 2 (DW: {dw_2:.2f})')
axs[2].grid(True)




plt.tight_layout()
plt.show()